In [1]:
import ibis
from ibis import _
import ibis.selectors as s
import polars as pl

In [2]:
ibis.options.interactive = True

con = ibis.duckdb.connect()

In [39]:
enr_old = (
    con.read_csv("data/clean/enrollment_2012-13_2022-23.csv")
    .filter(_.YEAR >= 2013)
    .select(
        year=_.YEAR,
        academic_year=_.YEAR_LONG,
        county_code=_.COUNTY_CODE,
        district_code=_.DISTRICT_CODE,
        county_name=_.COUNTY_NAME,
        district_name=_.DISTRICT_NAME,
        pk=_.PRE_K_ENROLLMENT,
        k12=_.K_12_ENROLLMENT,
    )
    # .to_pandas()
)

In [ ]:
enr_new = pl.read_excel(
    "data/raw/enrollment_2324.xlsx",
    sheet_name="District",
    read_options={"header_row": 2},
)

enr_new = (
    ibis.memtable(enr_new)
    .rename("snake_case")
    .filter(_.county_code != "End of worksheet")
    .mutate(
        year=2023,
        # This fails. I can't find any way to simply create a column with just 1 value.
        # Ibis is still not ready for primetime.
        academic_year="2023-24",
        pk=_.pre_k_halfday + _.pre_k_fullday,
    )
    .mutate(k12=_.total_enrollment - _.pk,)
    .select(
        _.county_code,
        _.district_code,
        _.county_name,
        _.district_name,
        _.pk,
        _.k12,
    )
    # .to_pandas()
)

AttributeError: module 'ibis' has no attribute 'Int64'. 

In [ ]:
# This can't workaround the above problem bcs union requires similar schemas. AFAIK
# there's no Ibis equivalent to dplyr::bind_rows.
enr = (
    enr_new.union(enr_old)
    .mutate(
        year=_.year.fill_null(2023),
        academic_year=_.academic_year.fill_null("2023-24"),
    )
    .order_by(_.year, _.county_code, _.district_code)
)

RelationError: Table schemas must be equal for set operations.
Columns missing from the left:
ibis.Schema {
  year           int64
  academic_year  string
}.

In [27]:
con.to_csv(enr, "data/clean/enrollment_2023-24.csv")